Example 4 - Unsupervised Learning — K-Means Clustering

This notebook demonstrates **clustering**, where the dataset has input features but **no target/class label** for the algorithm to learn.

### Example problem
Group students according to similarities in:
- Attendance
- GPA
- Study Hours
- Failed Subjects

### Learning flow
**Dataset → Inspect → Select Features → Scale → Explore K → Fit K-Means → Evaluate → Interpret → Label Clusters → Predict Cluster for New Student**

> Cluster numbers such as `Cluster 0` and `Cluster 1` have no inherent meaning. Human-readable labels are assigned only after examining cluster centers and member characteristics.


In [ ]:
# PHASE 1 — IMPORT LIBRARIES
# GUIDE: Run this cell first.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")


## Phase 2 — Create a Synthetic Student Dataset

For teaching purposes, we will generate **300 unique student records**. The dataset does not contain an `At Risk` target. K-Means must discover groups from the numerical characteristics themselves.


In [ ]:
# PHASE 2 — CREATE SYNTHETIC DATASET
# GUIDE: random_state/seed makes the demonstration reproducible.

rng = np.random.default_rng(42)
n = 300

df = pd.DataFrame({
    "Student_ID": [f"S{i:03d}" for i in range(1, n+1)],
    "Attendance": np.clip(rng.normal(82, 11, n), 45, 100).round(1),
    "GPA": np.clip(rng.normal(2.25, 0.55, n), 1.0, 4.0).round(2),
    "Study_Hours": np.clip(rng.normal(9, 4, n), 0, 24).round(1),
    "Failed_Subjects": np.clip(rng.poisson(0.9, n), 0, 5)
})

# Introduce relationships so the exercise has interpretable structure.
low_att = df["Attendance"] < 72
df.loc[low_att, "GPA"] = np.clip(
    df.loc[low_att, "GPA"] + rng.uniform(0.25, 0.75, low_att.sum()), 1, 4
).round(2)
df.loc[low_att, "Failed_Subjects"] = np.clip(
    df.loc[low_att, "Failed_Subjects"] + rng.integers(1, 3, low_att.sum()), 0, 5
)

high_study = df["Study_Hours"] >= 12
df.loc[high_study, "GPA"] = np.clip(
    df.loc[high_study, "GPA"] - rng.uniform(0.15, 0.45, high_study.sum()), 1, 4
).round(2)

print("Dataset shape:", df.shape)
display(df.head(10))


## Phase 3 — Inspect the Dataset

Before clustering, check missing values, duplicates, ranges, and descriptive statistics. Unlike supervised classification, there is **no `y` target variable** here.


In [ ]:
# PHASE 3 — DATA INSPECTION

print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe())


## Phase 4 — Select Features and Scale Them

K-Means uses distances. A feature with a larger numerical scale can dominate the distance calculation, so standardization is normally important.

We will cluster using four features.


In [ ]:
# PHASE 4 — DEFINE X AND SCALE FEATURES
# GUIDE: There is X, but no supervised-learning target y.

features = [
    "Attendance",
    "GPA",
    "Study_Hours",
    "Failed_Subjects"
]

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features used:", features)
print("Original X shape:", X.shape)
print("Scaled X shape:", X_scaled.shape)

display(X.head())


## Phase 5 — Explore the Number of Clusters

For K-Means, the number of clusters **K** is a hyperparameter.

We can examine:
- **Inertia / Elbow Method** — within-cluster squared distance.
- **Silhouette Score** — how well observations fit their assigned cluster relative to neighboring clusters.

A higher silhouette score is generally better, but the final choice should also consider **interpretability and the research objective**.


In [ ]:
# PHASE 5 — COMPARE POSSIBLE K VALUES

k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
        max_iter=300
    )
    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

k_comparison = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette_Score": silhouettes
})

display(k_comparison.round(4))

plt.figure(figsize=(7, 4))
plt.plot(k_comparison["K"], k_comparison["Inertia"], marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(k_comparison["K"], k_comparison["Silhouette_Score"], marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for Candidate K")
plt.show()

best_k_silhouette = int(
    k_comparison.loc[k_comparison["Silhouette_Score"].idxmax(), "K"]
)
print("K with highest silhouette score:", best_k_silhouette)


## Phase 6 — Interactive K-Means Settings

Now the student can experiment with K-Means hyperparameters.

### Inputs
- `n_clusters` — requested number of clusters.
- `max_iter` — maximum number of optimization iterations.
- `n_init` — number of different centroid initializations; the best run is retained.
- `random_state` — reproducibility.

This lets students rerun the model and observe whether the clusters and evaluation change.


In [ ]:
# PHASE 6 — USER INPUT FOR K-MEANS HYPERPARAMETERS

def ask_int(prompt, default, minimum=1):
    while True:
        raw = input(f"{prompt} [default={default}]: ").strip()
        if raw == "":
            return default
        try:
            value = int(raw)
            if value >= minimum:
                return value
            print(f"Please enter a value >= {minimum}.")
        except ValueError:
            print("Please enter a whole number.")

user_k = ask_int("Number of clusters (K)", best_k_silhouette, 2)
user_max_iter = ask_int("Maximum iterations", 300, 1)
user_n_init = ask_int("Number of initializations (n_init)", 10, 1)
user_random_state = ask_int("Random state", 42, 0)

print("\nSELECTED SETTINGS")
print("K:", user_k)
print("max_iter:", user_max_iter)
print("n_init:", user_n_init)
print("random_state:", user_random_state)


## Phase 7 — Fit the User-Configured K-Means Model

K-Means repeatedly assigns observations to the nearest centroid and updates the centroids until convergence or the iteration limit is reached.


In [ ]:
# PHASE 7 — FIT K-MEANS

kmeans = KMeans(
    n_clusters=user_k,
    max_iter=user_max_iter,
    n_init=user_n_init,
    random_state=user_random_state
)

cluster_labels = kmeans.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["Cluster"] = cluster_labels

sil = silhouette_score(X_scaled, cluster_labels)

print("K-Means completed.")
print("Number of clusters:", user_k)
print("Iterations actually performed:", kmeans.n_iter_)
print("Inertia:", round(kmeans.inertia_, 3))
print("Silhouette score:", round(sil, 3))

print("\nCluster sizes:")
display(
    df_clustered["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Students")
    .to_frame()
)


## Phase 8 — Interpret the Cluster Centers

K-Means centroids are currently in standardized units. We convert them back to the original feature scales so the results are easier to interpret.

For example, a cluster might show:
- higher attendance,
- lower GPA values,
- more study hours,
- fewer failed subjects.

Because the synthetic dataset uses the Philippine-style GPA example where **lower GPA values are better**, interpretation should take that direction into account.


In [ ]:
# PHASE 8 — CONVERT CENTROIDS BACK TO ORIGINAL UNITS

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)

centers_df = pd.DataFrame(
    centers_original,
    columns=features
)
centers_df.insert(0, "Cluster", range(user_k))

centers_df["Attendance"] = centers_df["Attendance"].round(1)
centers_df["GPA"] = centers_df["GPA"].round(2)
centers_df["Study_Hours"] = centers_df["Study_Hours"].round(1)
centers_df["Failed_Subjects"] = centers_df["Failed_Subjects"].round(2)

print("CLUSTER CENTERS IN ORIGINAL UNITS")
display(centers_df)


## Phase 9 — Suggested and User-Defined Cluster Labels

K-Means outputs only cluster numbers. The notebook therefore generates a **transparent suggested description** based on whether each centroid is above or below the overall dataset averages.

The student should inspect the centroid table and actual members before accepting a label. You can press **Enter** to keep the suggestion or type your own label.


## Phase 9B — Assign Meaningful Cluster Labels

The interface below automatically creates **one visible text box for every discovered cluster**.

Each box is pre-filled with the suggested description. You can:

1. keep the suggested label;
2. edit it into a shorter and more meaningful label; or
3. replace it completely.

For example, after examining the centroid and cluster members, you might use labels such as **Highly Engaged**, **Needs Academic Support**, or **Moderately Engaged**.

After editing the boxes, click **Apply Cluster Labels**. The labels will then be stored in `final_cluster_labels` and added to the clustered dataset.

> **Guide:** Cluster labels are interpretations, not outputs learned by K-Means. Always justify them using the centroid values and characteristics of the cluster members.


In [ ]:
# PHASE 9B — VISIBLE CLUSTER LABEL INPUT BOXES
# SELF-CONTAINED VERSION:
# This cell reconstructs the cluster centers and suggested labels if needed,
# so it will not fail simply because the immediately preceding label cell
# was skipped.

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ----- SAFETY CHECKS -----
required_names = ["df_clustered", "kmeans", "scaler", "X", "features", "user_k"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Required earlier clustering steps have not been run. "
        "Please run the notebook from Phase 1 through Phase 7 first. "
        "Missing: " + ", ".join(missing)
    )

# ----- RECREATE CENTROIDS IF NECESSARY -----
if "centers_df" not in globals():
    centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
    centers_df = pd.DataFrame(centers_original, columns=features)
    centers_df.insert(0, "Cluster", range(user_k))
    centers_df["Attendance"] = centers_df["Attendance"].round(1)
    centers_df["GPA"] = centers_df["GPA"].round(2)
    centers_df["Study_Hours"] = centers_df["Study_Hours"].round(1)
    centers_df["Failed_Subjects"] = centers_df["Failed_Subjects"].round(2)

# ----- ALWAYS BUILD SUGGESTED LABELS HERE -----
# GUIDE: This prevents NameError if the earlier suggestion cell was skipped.
overall = X.mean()
suggested_labels = {}

for _, row in centers_df.iterrows():
    c = int(row["Cluster"])
    descriptors = []

    descriptors.append(
        "Higher Attendance"
        if row["Attendance"] >= overall["Attendance"]
        else "Lower Attendance"
    )

    # In this teaching dataset, lower numerical GPA is stronger.
    descriptors.append(
        "Stronger GPA"
        if row["GPA"] <= overall["GPA"]
        else "Weaker GPA"
    )

    descriptors.append(
        "Higher Study Hours"
        if row["Study_Hours"] >= overall["Study_Hours"]
        else "Lower Study Hours"
    )

    descriptors.append(
        "More Failed Subjects"
        if row["Failed_Subjects"] > overall["Failed_Subjects"]
        else "Fewer Failed Subjects"
    )

    suggested_labels[c] = " | ".join(descriptors)

print("Suggested labels successfully generated:")
for c in range(user_k):
    print(f"Cluster {c}: {suggested_labels[c]}")

# ----- CREATE VISIBLE LABEL BOXES -----
cluster_label_boxes = {}

display(widgets.HTML(
    value="<h3>Assign Final Cluster Labels</h3>"
          "<p>Review the cluster profile, edit each label if desired, "
          "then click <b>Apply Cluster Labels</b>.</p>"
))

for c in range(user_k):
    center = centers_df.loc[centers_df["Cluster"] == c].iloc[0]

    display(widgets.HTML(value=(
        f"<b>Cluster {c}</b><br>"
        f"Attendance: {center['Attendance']:.1f} | "
        f"GPA: {center['GPA']:.2f} | "
        f"Study Hours: {center['Study_Hours']:.1f} | "
        f"Failed Subjects: {center['Failed_Subjects']:.2f}<br>"
        f"<i>Suggested label: {suggested_labels[c]}</i>"
    )))

    box = widgets.Text(
        value=suggested_labels[c],
        placeholder=f"Enter label for Cluster {c}",
        description=f"Label {c}:",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "70px"}
    )
    cluster_label_boxes[c] = box
    display(box)

apply_labels_button = widgets.Button(
    description="Apply Cluster Labels",
    button_style="success",
    icon="check"
)
label_output = widgets.Output()

def apply_cluster_labels(_):
    global final_cluster_labels, df_clustered

    final_cluster_labels = {}

    for c, box in cluster_label_boxes.items():
        label = box.value.strip()
        if label == "":
            label = suggested_labels[c]
            box.value = label
        final_cluster_labels[c] = label

    df_clustered["Cluster_Label"] = df_clustered["Cluster"].map(final_cluster_labels)

    with label_output:
        clear_output()
        print("CLUSTER LABELS APPLIED SUCCESSFULLY")
        print("-----------------------------------")
        for c in range(user_k):
            print(f"Cluster {c}: {final_cluster_labels[c]}")

        print("\nPreview:")
        display(
            df_clustered[
                ["Student_ID", "Cluster", "Cluster_Label"] + features
            ].head(15)
        )

apply_labels_button.on_click(apply_cluster_labels)

display(apply_labels_button)
display(label_output)

# GUIDE:
# 1. Edit any label you want.
# 2. Click Apply Cluster Labels.
# 3. Then proceed to Phase 10.


## Phase 10 — Examine Members of Each Cluster

Cluster interpretation should not rely on the centroid alone. Inspect actual members and descriptive statistics for each cluster.


In [ ]:
# PHASE 10 — CLUSTER PROFILES

# CHECK: Make sure the labels were applied in Phase 9B.
if "Cluster_Label" not in df_clustered.columns:
    raise RuntimeError(
        "Cluster labels have not been applied yet. "
        "Go back to Phase 9B, edit the visible label boxes, "
        "and click 'Apply Cluster Labels' before continuing."
    )


cluster_profiles = (
    df_clustered
    .groupby(["Cluster", "Cluster_Label"])[features]
    .mean()
    .round(2)
)

print("AVERAGE PROFILE OF EACH CLUSTER")
display(cluster_profiles)

for c in range(user_k):
    print("\n" + "=" * 80)
    print(f"CLUSTER {c}: {final_cluster_labels[c]}")
    print("=" * 80)

    members = df_clustered[df_clustered["Cluster"] == c]
    print("Number of students:", len(members))
    display(members.head(10))


## Phase 11 — Visualize the Clusters

A two-dimensional graph cannot show all four clustering dimensions at once. The graph below displays **Attendance vs. GPA** and uses the discovered cluster assignments for visual inspection.


In [ ]:
# PHASE 11 — VISUALIZE CLUSTERS
# GUIDE:
# This graph displays Attendance vs GPA as a 2D view of the clusters.
# IMPORTANT: K-Means used ALL FOUR selected features:
# Attendance, GPA, Study_Hours, and Failed_Subjects.
# The graph is only a 2D projection for easier interpretation.

# SAFETY CHECK:
# Make sure clustering has already been completed.
if "df_clustered" not in globals():
    raise RuntimeError(
        "Clustering results are not available. "
        "Please run Phases 1 through 9 first."
    )

# If cluster labels were not applied using the widget,
# create simple fallback labels so visualization can still run.
if "Cluster_Label" not in df_clustered.columns:
    print(
        "NOTE: Final cluster labels have not been applied. "
        "Using temporary labels for visualization."
    )

    if "suggested_labels" in globals():
        fallback_labels = {
            c: suggested_labels.get(c, f"Cluster {c}")
            for c in sorted(df_clustered["Cluster"].unique())
        }
    else:
        fallback_labels = {
            c: f"Cluster {c}"
            for c in sorted(df_clustered["Cluster"].unique())
        }

    df_clustered["Cluster_Label"] = (
        df_clustered["Cluster"].map(fallback_labels)
    )

# Build a reliable label dictionary directly from df_clustered.
plot_labels = (
    df_clustered[
        ["Cluster", "Cluster_Label"]
    ]
    .drop_duplicates()
    .set_index("Cluster")["Cluster_Label"]
    .to_dict()
)

# ---------------------------------------
# PLOT 1 — ATTENDANCE VS GPA
# ---------------------------------------

plt.figure(figsize=(10, 6))

cluster_ids = sorted(df_clustered["Cluster"].unique())

for c in cluster_ids:
    subset = df_clustered[df_clustered["Cluster"] == c]

    plt.scatter(
        subset["Attendance"],
        subset["GPA"],
        alpha=0.70,
        s=55,
        label=f"Cluster {c}: {plot_labels.get(c, f'Cluster {c}')}"
    )

# Add cluster centers to the graph.
# We use the centers already converted back to original units.
if "centers_df" in globals():
    plt.scatter(
        centers_df["Attendance"],
        centers_df["GPA"],
        marker="X",
        s=220,
        edgecolors="black",
        linewidths=1.5,
        label="Cluster Centroids"
    )

    # Label each centroid with its cluster number.
    for _, row in centers_df.iterrows():
        plt.annotate(
            f"C{int(row['Cluster'])}",
            (row["Attendance"], row["GPA"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold"
        )

plt.xlabel("Attendance (%)")
plt.ylabel("GPA")
plt.title("K-Means Clusters: Attendance vs GPA")

# Philippine-style GPA used in this exercise:
# lower numerical GPA represents stronger academic performance.
plt.gca().invert_yaxis()

plt.grid(alpha=0.25)

# Put legend outside the plot because user-defined labels can be long.
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

print("HOW TO READ THE GRAPH")
print("---------------------")
print("• Each point represents one student.")
print("• Points with the same cluster assignment appear as the same series.")
print("• X markers represent the cluster centroids.")
print("• C0, C1, C2, etc. identify the centroid of each cluster.")
print("• The GPA axis is inverted because lower numerical GPA is better")
print("  in this teaching example.")
print("• This graph uses only Attendance and GPA for visualization.")
print("• K-Means itself used all four selected clustering features.")


## Phase 12 — Interactive New Student Assignment

Although K-Means is unsupervised, a fitted K-Means model can assign a **new observation to the nearest learned centroid**.

This is not supervised classification. The model is determining which existing cluster the new student most closely resembles.


In [ ]:
# PHASE 12 — USER INPUT FOR A NEW STUDENT

def ask_float(prompt, minimum, maximum):
    while True:
        raw = input(prompt).strip()
        try:
            value = float(raw)
            if minimum <= value <= maximum:
                return value
            print(f"Enter a value from {minimum} to {maximum}.")
        except ValueError:
            print("Please enter a numeric value.")

attendance = ask_float("Attendance (0-100): ", 0, 100)
gpa = ask_float("GPA (1.0-4.0): ", 1.0, 4.0)
study_hours = ask_float("Study hours per week (0-50): ", 0, 50)
failed_subjects = ask_float("Number of failed subjects (0-20): ", 0, 20)

new_student = pd.DataFrame([{
    "Attendance": attendance,
    "GPA": gpa,
    "Study_Hours": study_hours,
    "Failed_Subjects": failed_subjects
}])

# IMPORTANT: Apply the SAME scaler learned from the original dataset.
new_student_scaled = scaler.transform(new_student)

assigned_cluster = int(kmeans.predict(new_student_scaled)[0])
assigned_label = final_cluster_labels[assigned_cluster]

# Distance to every centroid provides useful interpretation.
distances = kmeans.transform(new_student_scaled)[0]

print("\nNEW STUDENT")
display(new_student)

print("Assigned Cluster:", assigned_cluster)
print("Cluster Label:", assigned_label)

distance_table = pd.DataFrame({
    "Cluster": range(user_k),
    "Label": [final_cluster_labels[c] for c in range(user_k)],
    "Distance_to_Centroid": distances
}).sort_values("Distance_to_Centroid")

print("\nDISTANCE TO EACH CLUSTER CENTROID")
display(distance_table.round(3))

# GUIDE:
# The smallest centroid distance determines the assigned cluster.
# These distances are not probabilities.


## Phase 13 — Student Experiment

Rerun **Phase 6 onward** and try different values such as `K = 2`, `3`, `4`, `5`, or `6`.

Observe:
- silhouette score;
- inertia;
- cluster sizes;
- centroid characteristics;
- interpretability;
- whether clusters become too broad or too fragmented;
- whether the labels still make sense.

Do not choose K solely because one numerical metric is highest. The selected clustering solution should also be meaningful for the research problem.


## Student Interpretation Guide

Answer briefly:

1. Why is there no target variable `y` in this exercise?
2. Why were the numerical features standardized before K-Means?
3. What does `K` represent?
4. What does inertia measure?
5. What does the silhouette score tell us?
6. Which K had the highest silhouette score?
7. Which K did you finally use, and why?
8. How many iterations did the final model perform?
9. Describe each cluster using its centroid and member characteristics.
10. Why do cluster numbers require interpretation before labels are assigned?
11. What happened when you changed `K`, `max_iter`, or `n_init`?
12. How does assigning a new student to a cluster differ from supervised classification?
13. What limitations should be considered before using these clusters for real student decisions?

### Thesis documentation reminder

For a clustering study, document the selected features, preprocessing/scaling, clustering algorithm, hyperparameters, method used to choose the number of clusters, internal evaluation metrics, cluster profiles, labeling procedure, visualizations, and limitations. Cluster labels are researcher interpretations and should be supported by the observed characteristics of the cluster members.
